In [1]:
from uuid import uuid4
import os 
import numpy as np

from openenv.core.env_server import Environment, Action, Observation, State

import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from model import F1Action, F1Observation, F1State
from Track import gps_to_segments, compute_distance

d:\RL_Hackathon\RL\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
class F1Env(Environment):
    def __init__(
        self,
        target_seg_len: int =50,
        kappa_straight: float = 0.007,
        ay_max: float = 30.0,
        hs_speed_mps: float = 50.0,
        curv_percentile: int = 90,
        dt: float = 0.0125,
        initial_soc: float = 1.0,
        initial_velocity: float = 0.0,
        initial_tire_temp: float = 95.0,
        base_mu: float = 1.6
    ):
        super().__init__()

        track_path = "../track/Melbourne.csv"
        if not os.path.exists(track_path):
            raise FileNotFoundError(f"Track file not found at {track_path}") 
        ## Augs for track segments
        self.target_seg_len = target_seg_len
        self.kappa_straight = kappa_straight
        self.ay_max = ay_max
        self.hs_speed_mps = hs_speed_mps
        self.curv_percentile = curv_percentile

        self._state = State(episode_id=str(uuid4()), step_count=0)
        self.x, self.y, self.segments = gps_to_segments(
            track_path,
            target_seg_len=self.target_seg_len,
            kappa_straight=self.kappa_straight,
            ay_max=self.ay_max,
            hs_speed_mps=self.hs_speed_mps,
            curv_percentile=self.curv_percentile
        )

        self._cum_dist = compute_distance(self.x, self.y)
        self._total_length = self._cum_dist[-1]        
        self._segment_bounds_m = []
		
        for seg in self.segments:
            start_idx = int(seg["start_idx "])
            end_idx = int(seg["end_idx"])
            start_m = float(self._cum_dist[start_idx])
            end_m = float(self._cum_dist[end_idx])
            self._segment_bounds_m.append((start_m, max(start_m, end_m)))
        self.segments_end_m = self._segment_bounds_m[-1][1] if self._segment_bounds_m else 0.0


        ## Physics logic params
        self.dt = dt

        self.soc = initial_soc
        self.initial_soc = initial_soc

        self.velocity = initial_velocity
        self.initial_velocity = initial_velocity

        self.tire_temp = initial_tire_temp
        self.initial_tire_temp = initial_tire_temp

        self.tire_wear = 0.0

        self.mu = base_mu
        self.base_mu = base_mu

        self.aero_mode = int(self.segments[0].get("aero_mode", 0))
        self.throttle = 0.0
        self.brake = 0.0
        self.slip_ratio = 0.0 # need to update this
        self.slip_angle_rad = 0.0 # need to update this
        self.steering = 0.0 # need to update this


        self._segment_idx = 0
        self.progress_m = 0.0
        self._step_count = 0
        self._episode_id = None
        self._done = False

        # self.state = self.state() # need to update this

    def reset(self) -> F1Observation:
        
        self.episode_id = str(uuid4())
        self._step_count = 0
        self._done = False


        self.progress_m = 0.0
        self.velocity = self.initial_velocity
        self.soc = self.initial_soc
        self.tire_wear = 0.0
        self.mu = self.base_mu
        self.tire_temp = self.initial_tire_temp
        self._segment_idx = 0
        self.aero_mode = int(self.segments[0].get("aero_mode", 0))


        # self.state = self.state() # need to update this

    def step(self, action: Action) -> F1Observation:
        
        if self._done:
            pass
            # obs = self._get_observation()
            # exit

        throttle_cmd = float(np.clip(action.throttle, 0.0, 1.0))
        brake_cmd = float(np.clip(action.brake, 0.0, 1.0))
        steering_cmd = float(np.clip(action.steering, -1.0, 1.0))
        aero_mode_cmd = int(np.clip(action.aero_mode, 0, 1))
        regen_intensity = float(np.clip(action.regen_intensity, 0.0, 1.0))
        deploy_level = float(np.clip(action.deploy_level, 0.0, 1.0))

    @property
    def state(self) -> State:
        pass